In [1]:
from astropy.io import fits
from src.misc import *
from src.SPECTRUM import Spectrum
from src.FITSPECTRUM import FitSpectrum
from src.DP import DP
import matplotlib.pyplot as plt
import numpy as np
from astropy.table import Table

In [2]:
spectra_data    = fits.open('/Users/hyp0515/data/0715_Spring_BGS_ALL_trimmed.fits')
cigale_data     = fits.open('/Users/hyp0515/data/IronPhysProp_v1.2_extracted.fits')
fastspecfit     = fits.open('/Users/hyp0515/data/0715_Spring_half_BGS_BRIGHT_catalog_fastspecfit.fits')

In [3]:
x = np.linspace(-4, 1, 500)
sf_boundary = 0.61/(x-0.05) + 1.30
comp_boundary = 0.61/(x-0.47) + 1.19
liner_boundary = 1.05*x + 0.45

In [4]:
DP = DP()
dps_df, model_1comp_dps, left_2comp, right_2comp = DP.extract_fits_data('./catalogs/dp_catalog_test.fits')
# nbcs_df, model_1comp, _, _ = DP.extract_fits_data('./catalogs/nbcs_catalog.fits')
# cs_df, model_1comp_cs, _, _ = DP.extract_fits_data('./catalogs/cs_catalog.fits')

In [5]:
DP_SPECTRA = Spectrum(spectra_data, cigale_data, fastspecfit, load_targetID=dps_df['TARGETID'].to_list())
DP_SPECTRA = DP_SPECTRA.stack_data()
DP_SPECTRA = DP_SPECTRA.shift_to_rest_frame()

# NBCS_SPECTRA = Spectrum(spectra_data, cigale_data, fastspecfit, load_targetID=nbcs_df['TARGETID'].to_list())
# NBCS_SPECTRA = NBCS_SPECTRA.stack_data()
# NBCS_SPECTRA = NBCS_SPECTRA.shift_to_rest_frame()

# CS_SPECTRA = Spectrum(spectra_data, cigale_data, fastspecfit, load_targetID=cs_df['TARGETID'].to_list())
# CS_SPECTRA = CS_SPECTRA.stack_data()
# CS_SPECTRA = CS_SPECTRA.shift_to_rest_frame()

In [6]:
def bpt_classification(lam, flux, sigma, offset):
    
    unavailable_lines = []
    line_fluxes = []
    for i, lam0 in enumerate([Hbeta_rest[0], OIII_rest[1], Halpha_rest[0], NII_rest[1]]):
        line_flux = np.max(flux[(lam>lam0*(1+offset)-0.8) & (lam<lam0*(1+offset)+0.8)])
        line_noise = np.median(sigma[(lam>lam0*(1+offset)-0.8) & (lam<lam0*(1+offset)+0.8)])
        if line_flux < 3*line_noise:
            unavailable_lines.append(i)
            line_fluxes.append(line_noise)
        else:
            line_fluxes.append(line_flux)
        if len(unavailable_lines) > 1:
            return 'uncertain', []

    oiii_hbeta = np.log10(line_fluxes[1]/line_fluxes[0])
    nii_halpha = np.log10(line_fluxes[3]/line_fluxes[2])

    sf_boundary = 0.61/(nii_halpha-0.05)+1.30
    comp_boundary = 0.61/(nii_halpha-0.47)+1.19
    liner_boundary = 1.05*nii_halpha + 0.45
    if (sf_boundary > oiii_hbeta or comp_boundary > oiii_hbeta) and (nii_halpha < 0.47):
        if (sf_boundary > oiii_hbeta) and (nii_halpha < 0.05):
            return 'SF', line_fluxes
        else:
            return 'COMP', line_fluxes
    elif (sf_boundary < oiii_hbeta or comp_boundary < oiii_hbeta):
        if liner_boundary > oiii_hbeta:
            return 'LINER', line_fluxes
        else:
            return 'AGN', line_fluxes
    else:
        return 'uncertain', line_fluxes


In [7]:
classification_map_2comp = {
    2: 'double SF', 8: 'double COMP', 32: 'double AGN', 128: 'double LINER',
    5: 'SF+COMP', 17: 'SF+AGN', 65: 'SF+LINER',
    20: 'COMP+AGN', 68: 'COMP+LINER', 80: 'AGN+LINER',
    257: 'SF+uncertain', 260: 'COMP+uncertain', 272: 'AGN+uncertain', 320: 'LINER+uncertain',
    512: 'unclassified'
}

# Ensure the new column exists
dps_df['BPT_class'] = 0

dps_blue_fluxes = []
dps_blue_ids = []

dps_red_fluxes = []
dps_red_ids = []

for i in range(len(DP_SPECTRA.data_stack)):
    lam = DP_SPECTRA.data_stack[i, 0, :]
    sigma = 1/np.sqrt(np.abs(DP_SPECTRA.data_stack[i, 2, :]))

    bpt_class_val = 0
    models = [left_2comp[i], right_2comp[i]]
    offsets = [dps_df.iloc[i]['dv_l']/c, dps_df.iloc[i]['dv_r']/c]

    for j, (model, offset) in enumerate(zip(models, offsets)):
        classification, line_fluxes = bpt_classification(lam, model, sigma, offset)
        if classification == 'SF':
            bpt_class_val += 1
        elif classification == 'COMP':
            bpt_class_val += 4
        elif classification == 'AGN':
            bpt_class_val += 16
        elif classification == 'LINER':
            bpt_class_val += 64
        elif classification == 'uncertain':
            bpt_class_val += 256
        
        if line_fluxes != []:
            if j == 0:
                dps_blue_fluxes.append(line_fluxes)
                dps_blue_ids.append(dps_df.iloc[i]['TARGETID'])
            else:
                dps_red_fluxes.append(line_fluxes)
                dps_red_ids.append(dps_df.iloc[i]['TARGETID'])

    # Use the map to get the classification string, with a default for unhandled cases
    # dps_df.at[i, 'BPT_class'] = classification_map_2comp.get(bpt_class_val, 'unclassified')
    dps_df.at[i, 'BPT_class'] = bpt_class_val

# Display the counts of each classification
print(dps_df['BPT_class'].value_counts())


/var/folders/_b/sl_t4k5539781f29qf723b080000gn/T/ipykernel_23519/2742641853.py:20: RuntimeWarning: divide by zero encountered in divide
  sigma = 1/np.sqrt(np.abs(DP_SPECTRA.data_stack[i, 2, :]))
/var/folders/_b/sl_t4k5539781f29qf723b080000gn/T/ipykernel_23519/3670727628.py:16: RuntimeWarning: divide by zero encountered in log10
  oiii_hbeta = np.log10(line_fluxes[1]/line_fluxes[0])
/var/folders/_b/sl_t4k5539781f29qf723b080000gn/T/ipykernel_23519/3670727628.py:17: RuntimeWarning: divide by zero encountered in log10
  nii_halpha = np.log10(line_fluxes[3]/line_fluxes[2])


BPT_class
512    11350
257     5946
2       5818
5        605
260      597
272      572
17       386
32       196
8        179
20       144
320        6
80         5
68         2
65         1
Name: count, dtype: int64


In [8]:
DP.get_catalog(df=dps_df, model_1comp=model_1comp_dps, left_2comp=left_2comp, right_2comp=right_2comp, fname='./catalogs/dps_classified.fits')

In [9]:
# SF_class = ['double SF', 'SF+COMP', 'SF+AGN', 'SF+LINER', 'SF+uncertain']
# AGN_class = ['double AGN', 'COMP+AGN', 'SF+AGN', 'AGN+LINER', 'AGN+uncertain']
# COMP_class = ['double COMP', 'SF+COMP', 'COMP+AGN', 'COMP+LINER', 'COMP+uncertain']
# LINER_class = ['double LINER', 'SF+LINER', 'COMP+LINER', 'AGN+LINER', 'LINER+uncertain']


In [10]:
# sf_dps = dps_df[dps_df['BPT_class'].isin(['double SF', 'SF+COMP', 'SF+uncertain'])]
# agn_dps = dps_df[dps_df['BPT_class'].isin(['double AGN', 'AGN+LINER', 'COMP+AGN', 'COMP+LINER', 'AGN+uncertain'])]

In [11]:
# classification_map_1comp = {
#     2: 'SF', 8: 'COMP', 32: 'AGN', 128: 'LINER',
#     512: 'unclassified'
# }

# # Ensure the new column exists
# nbcs_df['BPT_class'] = ''

# nbcs_fluxes = []
# nbcs_ids = []
# for i in range(len(NBCS_SPECTRA.data_stack)):
#     lam = NBCS_SPECTRA.data_stack[i, 0, :]
#     sigma = 1/np.sqrt(np.abs(NBCS_SPECTRA.data_stack[i, 2, :]))

#     bpt_class_val = 0

#     classification, line_fluxes = bpt_classification(lam, model_1comp[i], sigma, 0)
#     if classification == 'SF':
#         bpt_class_val += 2
#     elif classification == 'COMP':
#         bpt_class_val += 8
#     elif classification == 'AGN':
#         bpt_class_val += 32
#     elif classification == 'LINER':
#         bpt_class_val += 128
#     elif classification == 'uncertain':
#         bpt_class_val += 512
    
#     if line_fluxes != []:
#         nbcs_fluxes.append(line_fluxes)
#         nbcs_ids.append(nbcs_df.iloc[i]['TARGETID'])

#     # Use the map to get the classification string, with a default for unhandled cases
#     nbcs_df.at[i, 'BPT_class'] = classification_map_1comp.get(bpt_class_val, 'unclassified')

# # Display the counts of each classification
# print(nbcs_df['BPT_class'].value_counts())


In [12]:
# classification_map_1comp = {
#     2: 'SF', 8: 'COMP', 32: 'AGN', 128: 'LINER',
#     512: 'unclassified'
# }

# # Ensure the new column exists
# cs_df['BPT_class'] = ''
# cs_fluxes = []
# cs_ids = []
# for i in range(len(CS_SPECTRA.data_stack)):
#     lam = CS_SPECTRA.data_stack[i, 0, :]
#     sigma = 1/np.sqrt(np.abs(CS_SPECTRA.data_stack[i, 2, :]))

#     bpt_class_val = 0

#     classification, line_fluxes = bpt_classification(lam, model_1comp_cs[i], sigma, 0)
#     if classification == 'SF':
#         bpt_class_val += 2
#     elif classification == 'COMP':
#         bpt_class_val += 8
#     elif classification == 'AGN':
#         bpt_class_val += 32
#     elif classification == 'LINER':
#         bpt_class_val += 128
#     elif classification == 'uncertain':
#         bpt_class_val += 512

#     if line_fluxes != []:
#         cs_fluxes.append(line_fluxes)
#         cs_ids.append(cs_df.iloc[i]['TARGETID'])

#     # Use the map to get the classification string, with a default for unhandled cases
#     cs_df.at[i, 'BPT_class'] = classification_map_1comp.get(bpt_class_val, 'unclassified')

# # Display the counts of each classification
# print(cs_df['BPT_class'].value_counts())


In [13]:
# sf_dps['delta_dv'] = sf_dps['dv_r'] - sf_dps['dv_l']
# print(len(sf_dps[sf_dps['Z']<0.25]))
# sf_dps[sf_dps['Z']<0.25].sort_values(by=['delta_dv'], ascending=[False]).head(10)

In [14]:
# agn_dps['delta_dv'] = agn_dps['dv_r'] - agn_dps['dv_l']
# print(len(agn_dps[agn_dps['Z']<0.25]))
# agn_dps[agn_dps['Z']<0.25].sort_values(by=['delta_dv'], ascending=[False]).head(10)

In [15]:
# sf_dps.to_csv('sf_dps.csv')
# agn_dps.to_csv('agn_dps.csv')

In [16]:
# fig, axes = plt.subplots(1, 4, figsize=(25, 6), sharey=True)
# plt.subplots_adjust(wspace=0.0, left=0.05, right=0.99, top=0.9, bottom=0.1)


# n_cs = len(cs_fluxes)
# ssfr_cs = cs_df[cs_df['TARGETID'].isin(cs_ids)]['LOGSFR'] - cs_df[cs_df['TARGETID'].isin(cs_ids)]['LOGM']
# cs_fluxes = np.array(cs_fluxes)
# oiii_hbeta = np.log10(cs_fluxes[:, 1]/cs_fluxes[:, 0])
# nii_halpha = np.log10(cs_fluxes[:, 3]/cs_fluxes[:, 2])
# axes[0].scatter(nii_halpha, oiii_hbeta, alpha=0.3, c=ssfr_cs, cmap='jet', vmin=-13, vmax=-8)
# axes[0].text(-1.8, 1.2, f'CS\n(N={n_cs})', fontsize=14)


# n_nbcs = len(nbcs_fluxes)
# nbcs_fluxes = np.array(nbcs_fluxes)
# ssfr_nbcs = nbcs_df[nbcs_df['TARGETID'].isin(nbcs_ids)]['LOGSFR'] - nbcs_df[nbcs_df['TARGETID'].isin(nbcs_ids)]['LOGM']
# oiii_hbeta = np.log10(nbcs_fluxes[:, 1]/nbcs_fluxes[:, 0])
# nii_halpha = np.log10(nbcs_fluxes[:, 3]/nbcs_fluxes[:, 2])
# axes[1].scatter(nii_halpha, oiii_hbeta, alpha=0.3, c=ssfr_nbcs, cmap='jet', vmin=-13, vmax=-8)
# axes[1].text(-1.8, 1.2, f'NBCS\n(N={n_nbcs})', fontsize=14)


# n_dps_left = len(dps_blue_fluxes)
# dps_blue_fluxes = np.array(dps_blue_fluxes)
# ssfr_blue = dps_df[dps_df['TARGETID'].isin(dps_blue_ids)]['LOGSFR'] - dps_df[dps_df['TARGETID'].isin(dps_blue_ids)]['LOGM']
# oiii_hbeta = np.log10(dps_blue_fluxes[:, 1]/dps_blue_fluxes[:, 0])
# nii_halpha = np.log10(dps_blue_fluxes[:, 3]/dps_blue_fluxes[:, 2])
# axes[2].scatter(nii_halpha, oiii_hbeta, alpha=0.3, c=ssfr_blue, cmap='jet', vmin=-13, vmax=-8)
# axes[2].text(-1.8, 1.2, f'DP Blue\n(N={n_dps_left})', fontsize=14)
        
# n_dps_right = len(dps_red_fluxes)
# dps_red_fluxes = np.array(dps_red_fluxes)
# ssfr_red = dps_df[dps_df['TARGETID'].isin(dps_red_ids)]['LOGSFR'] - dps_df[dps_df['TARGETID'].isin(dps_red_ids)]['LOGM']
# oiii_hbeta = np.log10(dps_red_fluxes[:, 1]/dps_red_fluxes[:, 0])
# nii_halpha = np.log10(dps_red_fluxes[:, 3]/dps_red_fluxes[:, 2])
# sc = axes[3].scatter(nii_halpha, oiii_hbeta, alpha=0.3, c=ssfr_red, cmap='jet', vmin=-13, vmax=-8)
# axes[3].text(-1.8, 1.2, f'DP Red\n(N={n_dps_right})', fontsize=14)

# for i in range(4):
#     ax = axes[i]
#     ax.plot(x[(x<0.05)|((sf_boundary<comp_boundary)&(sf_boundary<liner_boundary))], sf_boundary[(x<0.05)|((sf_boundary<comp_boundary)&(sf_boundary<liner_boundary))], color='blue', linewidth=3, linestyle='--')
#     ax.plot(x[(x<0.47)&(comp_boundary>sf_boundary)|(comp_boundary<liner_boundary)], comp_boundary[(x<0.47)&(comp_boundary>sf_boundary)|(comp_boundary<liner_boundary)], color='red', linewidth=3)
#     ax.plot(x[(x>0)|(liner_boundary>comp_boundary)], liner_boundary[(x>0)|(liner_boundary>comp_boundary)], color='green', linewidth=3)
#     ax.set_xlabel('log([NII]6584/Hα)')
#     if i == 0:
#         ax.set_ylabel('log([OIII]5007/Hβ)')
#     ax.set_xlim(-2, .7)
#     ax.set_ylim(-1.5, 1.7)

# fig.subplots_adjust(right=0.9)
# cbar_ax = fig.add_axes([0.91, 0.10, 0.01, 0.8])
# cbar = fig.colorbar(sc, cax=cbar_ax)
# cbar.set_label('log(sSFR)')
# # plt.tight_layout()
# plt.savefig('./figures/Jan22/bpt_diagram.png')
# plt.show()

In [17]:
# classification_map_1comp = {
#     2: 'SF', 8: 'COMP', 32: 'AGN', 128: 'LINER',
#     512: 'unclassified'
# }

# # Ensure the new column exists
# dps_df['BPT_class_1comp'] = ''

# dps_fluxes_1comp = []
# dps_ids_1comp = []
# for i in range(len(DP_SPECTRA.data_stack)):
#     lam = DP_SPECTRA.data_stack[i, 0, :]
#     sigma = 1/np.sqrt(np.abs(DP_SPECTRA.data_stack[i, 2, :]))

#     bpt_class_val = 0

#     classification, line_fluxes = bpt_classification(lam, model_1comp_dps[i], sigma, 0)
#     if classification == 'SF':
#         bpt_class_val += 2
#     elif classification == 'COMP':
#         bpt_class_val += 8
#     elif classification == 'AGN':
#         bpt_class_val += 32
#     elif classification == 'LINER':
#         bpt_class_val += 128
#     elif classification == 'uncertain':
#         bpt_class_val += 512
    
#     if line_fluxes != []:
#         dps_fluxes_1comp.append(line_fluxes)
#         dps_ids_1comp.append(dps_df.iloc[i]['TARGETID'])

#     # Use the map to get the classification string, with a default for unhandled cases
#     dps_df.at[i, 'BPT_class_1comp'] = classification_map_1comp.get(bpt_class_val, 'unclassified')

# # Display the counts of each classification
# print(dps_df['BPT_class_1comp'].value_counts())
